# Resolve a CSV batch

The entire input is validated before the first paid search. The output CSV is checkpointed after every completed product.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from dotenv import load_dotenv

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file()
)
os.chdir(ROOT)
load_dotenv(ROOT / ".env", override=False)

from product_url import Budgets, ProductInput, ProductURLResolver

print(f"Repository: {ROOT}")
print("SERPAPI_API_KEY configured:", bool(os.getenv("SERPAPI_API_KEY")))
print("PCA reasoning enabled:", os.getenv("PRODUCT_URL_REASONING_ENABLED", "true"))

## Paths and budgets

In [ ]:
INPUT_CSV = ROOT / "samples" / "products.csv"
OUTPUT_CSV = ROOT / "data" / "results" / "product_urls.csv"
BUDGETS = Budgets(searches=3, search_results=10, crawls=6, llm_calls=2)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

## Load and preflight every row

In [ ]:
source = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
source = source.rename(columns={
    "MAIN_TEXT": "main_text",
    "COUNTRY": "country_code",
    "EAN": "ean",
    "RETAILER": "retailer_name",
    "ROW_ID": "row_id",
})

missing = {"main_text", "country_code"} - set(source.columns)
if missing:
    raise ValueError(f"Missing mandatory columns: {sorted(missing)}")

for name in ("row_id", "ean", "retailer_name"):
    if name not in source.columns:
        source[name] = ""

PRODUCTS = [
    ProductInput(
        row_id=row.row_id or None,
        main_text=row.main_text,
        country_code=row.country_code,
        ean=row.ean or None,
        retailer_name=row.retailer_name or None,
    )
    for row in source.itertuples(index=False)
]

row_ids = [item.row_id for item in PRODUCTS]
if len(row_ids) != len(set(row_ids)):
    raise ValueError("row_id values must be unique")

print(f"Validated {len(PRODUCTS)} products before any SerpAPI call.")

## Run and checkpoint

In [ ]:
ROWS = []

async with ProductURLResolver(
    budgets=BUDGETS,
    artifact_root=ROOT / "data" / "artifacts",
) as resolver:
    for index, product in enumerate(PRODUCTS, start=1):
        print(f"[{index}/{len(PRODUCTS)}] {product.row_id} — {product.main_text[:80]}")
        result = await resolver.resolve(product)
        ROWS.append(result["output"])
        pd.DataFrame(ROWS).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"Saved: {OUTPUT_CSV}")

## Output

In [ ]:
OUTPUT = pd.read_csv(OUTPUT_CSV, dtype=str, keep_default_na=False)
OUTPUT

## Basic output contract

In [ ]:
delivered = OUTPUT["VALIDATION_STATUS"].isin(["VERIFIED", "REVIEW_REQUIRED"])
has_url = OUTPUT["PRODUCT_URL"].str.strip().ne("")
assert delivered.eq(has_url).all(), "Delivered rows must contain one URL; failed rows must not."
print(OUTPUT.groupby("VALIDATION_STATUS").size())